In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/data_wo_cols_20260912_212149.csv")

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"
]

X = df[features]
y = df["Machine failure"]

tree = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- Rotational speed [rpm] <= 1386.50
|   |--- Air temperature [K] <= 301.55
|   |   |--- Torque [Nm] <= 53.55
|   |   |   |--- Tool wear [min] <= 202.50
|   |   |   |   |--- class: 0
|   |   |   |--- Tool wear [min] >  202.50
|   |   |   |   |--- Air temperature [K] <= 300.65
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- Air temperature [K] >  300.65
|   |   |   |   |   |--- class: 0
|   |   |--- Torque [Nm] >  53.55
|   |   |   |--- Tool wear [min] <= 171.50
|   |   |   |   |--- Torque [Nm] <= 62.25
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- Torque [Nm] >  62.25
|   |   |   |   |   |--- class: 1
|   |   |   |--- Tool wear [min] >  171.50
|   |   |   |   |--- Tool wear [min] <= 188.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- Tool wear [min] >  188.50
|   |   |   |   |   |--- class: 1
|   |--- Air temperature [K] >  301.55
|   |   |--- Process temperature [K] <= 312.25
|   |   |   |--- Rotational speed [rpm] <= 1380.50
|   |   |   |   |--- Process te

In [ ]:
# si la temperatura del aire es menor igual a 301.55 , menor igual a 300.65
# si la velocidad rotacional es menor igual a 1386.5
# si el torque es menor igual a 53.55
# si el tool wear es mayor  a 202.5
# si se cumple entonces si hay una falla

#

In [5]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def extraer_reglas_falla(tree, feature_names, clase_objetivo=1):
    """Devuelve lista de reglas (texto) que terminan en la clase objetivo."""
    tree_ = tree.tree_
    reglas = []

    def recorrer(nodo, condiciones):
        # ¿Es hoja?
        if tree_.children_left[nodo] == tree_.children_right[nodo]:
            # Clase predicha = argmax de los conteos
            clase = np.argmax(tree_.value[nodo][0])
            if clase == clase_objetivo:
                reglas.append(" AND ".join(condiciones) if condiciones else "RAIZ")
            return

        feat = feature_names[tree_.feature[nodo]]
        umbral = tree_.threshold[nodo]

        # Rama izquierda: <= umbral
        condiciones.append(f"{feat} <= {umbral:.2f}")
        recorrer(tree_.children_left[nodo], condiciones)
        condiciones.pop()

        # Rama derecha: > umbral
        condiciones.append(f"{feat} > {umbral:.2f}")
        recorrer(tree_.children_right[nodo], condiciones)
        condiciones.pop()

    recorrer(0, [])
    return reglas

# Uso
reglas_falla = extraer_reglas_falla(tree, features, clase_objetivo=1)

for i, r in enumerate(reglas_falla, 1):
    print(f"R{i}: SI {r} -> FALLA (class 1)")

R1: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] <= 301.55 AND Torque [Nm] <= 53.55 AND Tool wear [min] > 202.50 AND Air temperature [K] <= 300.65 -> FALLA (class 1)
R2: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] <= 301.55 AND Torque [Nm] > 53.55 AND Tool wear [min] <= 171.50 AND Torque [Nm] > 62.25 -> FALLA (class 1)
R3: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] <= 301.55 AND Torque [Nm] > 53.55 AND Tool wear [min] > 171.50 AND Tool wear [min] <= 188.50 -> FALLA (class 1)
R4: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] <= 301.55 AND Torque [Nm] > 53.55 AND Tool wear [min] > 171.50 AND Tool wear [min] > 188.50 -> FALLA (class 1)
R5: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] > 301.55 AND Process temperature [K] <= 312.25 AND Rotational speed [rpm] <= 1380.50 AND Process temperature [K] <= 310.55 -> FALLA (class 1)
R6: SI Rotational speed [rpm] <= 1386.50 AND Air temperature [K] > 301.55 AND Process 